In [ ]:
import json
import numpy as np
from constants import *
import json

data = []
with open(GRADED_PROMPTS_AIRBENCH, "r") as f:
    for line in f:
        data.append(json.loads(line))

value_categories_set = set()
new_data = []
all_values = np.zeros((len(data)*11, NUM_VALENCED_VALUE_CATEGORIES))
for d_idx, d in enumerate(data):
    new_d = {"id": d_idx}
    for l in range(0, 11):
        level = f"level_{l/10:.1f}"
        try:
            value_categories = d[f"{level}_kaleido_top_k_valenced_topics"]
            value_categories_set.update(value_categories)
            valence_scores = d[f"{level}_kaleido_top_k_valenced_topics_valence_scores"]
            new_d[level] = {"value_categories": value_categories, "valance_scores": valence_scores}
            all_values[d_idx*11 + l, value_categories] = valence_scores
        except KeyError:
            new_d[level] = np.nan
    new_data.append(new_d)

with open(TOP_K_KALEIDO_TOPICS_JSONL, "w") as f:
    for new_d in new_data:
        f.write(json.dumps(new_d) + "\n")
np.save(TOP_K_KALEIDO_TOPICS_NPY, all_values)

In [ ]:
with open(TOP_K_KALEIDO_TOPICS_JSONL, "r") as f:
    topic_id_to_name = json.load(f)

for i in range(NUM_VALENCED_VALUE_CATEGORIES / 2):
    print(f'{i}: "[Value] {topic_id_to_name[str(i)]}",')
for i in range(NUM_VALENCED_VALUE_CATEGORIES / 2):
    print(f'{i+39}: "[Value] (negative) {topic_id_to_name[str(i)]}",')